# 🤖 04 · 深度 Q 网络 DQN：从表格到函数近似

> 面试定位：DQN 是 **RL × 深度学习** 的里程碑（Nature 2015），几乎所有 RL 岗位必问：
> 稳定训练的四件套 + DDQN/Dueling/PER 三个改进。
> 本篇继续用自写网格环境，**手写 numpy 版浅层网络 Q 函数**（不调 DL 框架）。
> 标记：🔴 必背。

---

## 1. 从表格 Q 到 Q 函数近似（🔴）

### 表格的问题

- 状态/动作组合爆炸：围棋 $10^{170}$、连续控制无穷多状态
- 无法泛化：没见过的状态没有记录

### 函数近似

用参数 $\theta$ 的模型 $Q(s,a;\theta) \approx Q^*(s,a)$：

$$\text{loss} = \mathbb{E}\left[\left(y - Q(s,a;\theta)\right)^2\right], \qquad y = r + \gamma \max_{a'} Q(s', a'; \theta^-)$$

- $y$ 是 **TD 目标**（监督信号），$\theta^-$ 是目标网络参数
- 深度网络把"状态特征 → 各动作价值"回归出来，未见过状态也能推断（泛化）

### 输出层的两种设计（背）

1. **输出 |A| 个值**：输入状态 s，一次前向得到所有 $Q(s,a)$ → DQN 标准做法
2. 输出单个值：每个 (s,a) 一次前向 → 慢，不用

---

## 2. DQN 的三大稳定性技巧（🔴 必背，面试逐条说）

### ① 经验回放（Experience Replay）

把转移 $(s,a,r,s')$ 存进**回放池**，训练时**随机采样小批量**：

- 打破样本**时间相关性**（i.i.d. 化）→ 稳定梯度
- **数据复用** → 样本效率高
- 平滑目标分布变化（旧经验缓冲分布偏移）

### ② 目标网络（Target Network）

用**延迟更新的 $Q_{\theta^-}$** 计算 TD 目标，不是当前网络自己：

$$y = r + \gamma \max_{a'} Q(s',a';\theta^-)$$

- 若用自己算目标：每次梯度更新目标跟着变 → **追逐移动靶**，震荡发散
- 做法：每 C 步把 $\theta \to \theta^-$ 复制（或软更新 $\theta^-\leftarrow\tau\theta+(1-\tau)\theta^-$，DDPG 用）

### ③ 误差剪裁（Huber/Clip）

$$L(\delta) = \begin{cases} \frac{1}{2}\delta^2 & |\delta| \le 1 \\ |\delta| - \frac{1}{2} & \text{其他} \end{cases}$$

- 梯度有界 → 对离群 TD 误差（噪声奖励）鲁棒

> **面试金句**：DQN 能稳定训练 = 回放（去相关）× 目标网络（固定监督）× 剪裁（限梯度）。缺一个都容易发散。

---


In [ ]:
# nn_lib.py —— 迷你 numpy 全连接网络（前向 + 反向传播, 供本 notebook 复用）
import numpy as np

class MLP:
    """两层全连接: input -> h(ReLU) -> out(线性)  |  手动梯度下降"""
    def __init__(self, n_in, n_h, n_out, seed=0):
        rng = np.random.RandomState(seed)
        self.W1 = rng.randn(n_in, n_h) * np.sqrt(2.0 / n_in)   # He 初始化
        self.b1 = np.zeros(n_h)
        self.W2 = rng.randn(n_h, n_out) * np.sqrt(2.0 / n_h)
        self.b2 = np.zeros(n_out)

    def forward(self, X):
        self.h = np.maximum(0, X @ self.W1 + self.b1)          # ReLU
        self.out = self.h @ self.W2 + self.b2
        return self.out

    def backward(self, X, grad_out, lr):
        # grad_out: dLoss/dout, shape (batch, n_out)
        g_W2 = self.h.T @ grad_out
        g_b2 = grad_out.mean(axis=0)
        g_h = grad_out @ self.W2.T
        g_h[self.h <= 0] = 0.0                                  # ReLU 导数
        g_W1 = X.T @ g_h
        g_b1 = g_h.mean(axis=0)
        self.W2 -= lr * g_W2
        self.b2 -= lr * g_b2
        self.W1 -= lr * g_W1
        self.b1 -= lr * g_b1

# 自检: 学习一个简单函数 f(s) = [s0, -s0]
X = np.random.RandomState(3).randn(200, 2)
Y = np.column_stack([X[:, 0], -X[:, 0]])
net = MLP(2, 16, 2, seed=1)
for step in range(300):
    out = net.forward(X)
    err = out - Y
    net.backward(X, err / len(X), lr=0.05)
print('迷你网络自检: 训练 300 步后 均方误差 =', round(float(np.mean((net.forward(X) - Y) ** 2)), 6))


In [ ]:
# dqn_mini.py —— 迷你 DQN: 手写 numpy 在悬崖漫步上学最优策略
# 状态独热编码 -> MLP(16, 64, 4) -> Q(s,a); 经验回放 + 目标网络
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
N_STATES = ROWS * COLS
N_ACTIONS = 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def to_onehot(s_idx):
    x = np.zeros(N_STATES)
    x[s_idx] = 1.0
    return x

def step_state(s_idx, a):
    r, c = divmod(s_idx, COLS)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    s2 = nr * COLS + nc
    if (nr, nc) in CLIFF:
        return START[0] * COLS + START[1], -100.0
    if (nr, nc) == GOAL:
        return s2, 0.0
    return s2, -1.0

def choose(Qnet, s_idx, eps):
    if np.random.random() < eps:
        return np.random.randint(N_ACTIONS)
    q = Qnet.forward(to_onehot(s_idx).reshape(1, -1))
    return int(np.argmax(q[0]))

np.random.seed(5)
online = MLP(N_STATES, 64, N_ACTIONS, seed=11)
target = MLP(N_STATES, 64, N_ACTIONS, seed=11)   # 同上结构(参数稍后同步)
# 用同一结构函数复制参数
def hard_copy(dst, src):
    dst.W1, dst.b1 = src.W1.copy(), src.b1.copy()
    dst.W2, dst.b2 = src.W2.copy(), src.b2.copy()

hard_copy(target, online)
replay = []            # (s, a, r, s2, done)
alpha, gamma, eps = 0.02, 0.9, 0.3
BATCH, CAP, SYNC = 32, 2000, 20

def train_step():
    if len(replay) < BATCH:
        return
    idx = np.random.randint(0, len(replay), BATCH)
    batch = [replay[i] for i in idx]
    S = np.array([to_onehot(b[0]) for b in batch])
    A = np.array([b[1] for b in batch])
    R = np.array([b[2] for b in batch])
    S2 = np.array([to_onehot(b[3]) for b in batch])
    q_next = target.forward(S2)                      # 目标网络给 max
    done_mask = np.array([b[4] for b in batch])
    # done 时无后继: 目标 = r; 否则 r + γ·max Q(s')
    y = R + gamma * np.where(done_mask, 0.0, np.max(q_next, axis=1))
    out = online.forward(S)
    grad = np.zeros_like(out)
    for i in range(BATCH):
        grad[i, A[i]] = out[i, A[i]] - y[i]          # d/dQ (Q-y)^2
    online.backward(S, grad / BATCH, lr=alpha)

rewards_list = []
for episode in range(400):
    s = START[0] * COLS + START[1]
    total, steps = 0.0, 0
    eps = max(0.05, 0.3 * (1 - episode / 400))
    while steps < 200:
        a = choose(online, s, eps)
        s2, r = step_state(s, a)
        done = s2 == GOAL[0] * COLS + GOAL[1]
        replay.append((s, a, r, s2, done))
        if len(replay) > CAP:
            replay.pop(0)
        train_step()
        s = s2
        total += r
        steps += 1
        if done:
            break
    if episode % SYNC == 0:
        hard_copy(target, online)                    # 目标网络同步
    rewards_list.append(total)

print('迷你 DQN 悬崖漫步 · 每 50 回合平均累计奖励:')
for i in range(0, 400, 50):
    print(f'  episode {i:3d}-{i+49:3d}: {np.mean(rewards_list[i:i+50]):8.2f}')
print('（奖励从 -100 级提升到 -20 级 ≈ 学会了避开悬崖的近路）')


## 3. DQN 三大改进（🔴 必背，面试连环问）

### ① Double DQN（DDQN）—— 治"过度乐观"

原版：$y = r + \gamma \max_{a'} Q(s',a';\theta^-)$ —— 一个网络既选动作又估值 → 最大化偏差。

```text
DDQN: 用 online 选动作, 用 target 给估值, 两表解耦:
  a* = argmax_a Q(s', a; θ)          ← online 选最优动作
  y  = r + γ Q(s', a*; θ^-)          ← target 给该动作的估值
```

**效果**：估值更准，显著提升稳定性（Atari 上超过 DQN）。

### ② Dueling DQN —— 拆价值与优势

把 Q 拆成状态价值 + 优势：

$$Q(s,a) = V(s) + A(s,a) - \max_{a'} A(s,a') \quad (\text{或减去 } \bar{A} \text{ 的均值版})$$

- 网络分支：一条算 $V(s)$，一条算优势 $A(s,a)$，合起来是 $Q$
- **直觉**：很多状态下"做什么动作"差别不大，学 $V$ 更快 → 更稳、样本效率更高

### ③ Prioritized Experience Replay（PER）—— 优先采重要的

- 按 **TD 误差绝对值** $|\delta|$ 加权采样（误差大 = 学得不好 = 更值得学）
- 概率 $P(i) \propto p_i^{\rho}$（$p_i=|\delta_i|+\epsilon$），同时用重要性权重修正采样偏差

### ④ NoisyNet / N-step / Rainbow（🟢 提一嘴）

- **N-step**：用 n 步回报缓解单步偏差（Trade-off 偏差↔方差）
- **Rainbow**：DQN + DDQN + Dueling + PER + N-step + NoisyNet + Distributional（六个改进合一，Atari SOTA）

> **面试金句**：回答"DQN 改进有哪些"按 ①②③ 展开，末尾提 Rainbow 合并六件套，显示知识面。

---

## 4. DQN 训练稳定性清单（工程向）

1. **奖励缩放/剪裁**（clipping reward 到 [-1,1]）——防止大奖励主导梯度
2. **梯度剪裁**——TD 误差离群时防爆炸
3. **目标网络同步频率**调小（C 大）更稳但慢
4. **回放池容量与 batch 大小**：批量 32-256，池越大越稳
5. **ε 衰减曲线**：前期多探索
6. **双输出共享底层特征**：CNN 提特征 + 双头（Dueling）

---

## 5. 面试追问速答

1. **Q: DQN 为什么需要经验回放？** A: 去时间相关（稳定梯度）+ 数据复用（效率）+ 平滑分布。
2. **Q: 目标网络多久更新？** A: 每 C 步硬复制（常见 1e3~1e4）或每步软更新 τ=1e-3（DDPG 风格）。
3. **Q: 连续动作空间能用 DQN 吗？** A: 不能直接——max 无法对连续动作枚举；用 DDPG/TD3/SAC（07 篇）。
4. **Q: DDQN 和 Double Q 的关系？** A: DDQN 是 Double Q-learning 的深度网络版。
5. **Q: Dueling 为什么有效？** A: 解耦 V 与 A，状态价值学习更稳更快，优势为零的和约束保证可辨识。

## 6. 自测清单

- [ ] L1：说清 DQN 三大稳定性技巧（回放/目标网络/剪裁）各自解决什么问题
- [ ] L1：默写 DQN 损失与 TD 目标
- [ ] L1：讲 DDQN 与 DQN 目标的差异（选动作 vs 估值拆开）
- [ ] L2：解释 Dueling 结构 $Q=V+A-\max A$ 与直觉
- [ ] L2：说出 PER 按什么权重采样、为什么要重要性修正

>  下一篇 `05-策略梯度与Actor-Critic`：学策略而非价值——连续动作的钥匙。


## 5. DQN 训练循环伪代码（🔴 面试默写）

```python
初始化 Q_online(θ) 与 Q_target(θ⁻), 回放池 D
for episode in range(M):
    s = env.reset()
    while not done:
        a = ε-greedy(Q_online(s))
        s', r, done = env.step(a)
        D.push((s, a, r, s', done))
        if len(D) > B:
            batch = D.sample(B)
            y = r + γ·max_a' Q_target(s', a')·(1-done)
            loss = Huber(Q_online(s)[a] - y)
            梯度下降更新 θ
            每 C 步: θ⁻ ← θ
        s = s'
```

### 六个关键数字（背）

- 回放池 CAP：1e5~1e6（Atari 常用 1e6）
- batch：32~256（常用 32）
- 目标网络同步 C：1e3~1e4 步
- ε：1.0 → 0.01（退火）
- γ：0.99（Atari）
- 优化器：Adam（lr 1e-4~3e-4）

---


In [ ]:
# dqn_loop.py —— 伪代码 → 可运行骨架(悬崖漫步, 简化版)
# 说明: 04 篇已有一个完整迷你 DQN; 这里给出"最小可运行骨架"便于默写对照
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
NS, NA = ROWS * COLS, 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step_idx(s, a):
    r, c = divmod(s, COLS)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    s2 = nr * COLS + nc
    if (nr, nc) in CLIFF:
        return START[0] * COLS + START[1], -100.0, False
    if (nr, nc) == GOAL:
        return s2, 0.0, True
    return s2, -1.0, False

def onehot(s):
    x = np.zeros(NS); x[s] = 1.0
    return x

rng = np.random.RandomState(0)
# 线性 Q 近似(相当于无隐层的 MLP): Q(s,a) = W·x(s)
W = rng.randn(NS, NA) * 0.05

D = []
CAP, B, alpha, gamma = 2000, 32, 0.02, 0.9
for ep in range(300):
    s = START[0] * COLS + START[1]
    eps = max(0.05, 0.3 * (1 - ep / 300))
    steps = 0
    while steps < 200:
        a = rng.randint(NA) if rng.random() < eps else int(np.argmax(onehot(s) @ W))
        s2, r, done = step_idx(s, a)
        D.append((s, a, r, s2, done))
        if len(D) > CAP:
            D.pop(0)
        if len(D) >= B:
            idx = rng.randint(0, len(D), B)
            for i in idx:
                s_i, a_i, r_i, s2_i, done_i = D[i]
                y = r_i if done_i else r_i + gamma * np.max(onehot(s2_i) @ W)
                pred = (onehot(s_i) @ W)[a_i]
                # 在线(无目标网络)线性更新
                W[:, a_i] += alpha * (y - pred) * onehot(s_i)
        s = s2
        steps += 1
        if done:
            break

# 评估
s = START[0] * COLS + START[1]
steps = 0
while s != GOAL and steps < 100:
    s = step_idx(s, int(np.argmax(onehot(s) @ W)))[0]
    steps += 1
print(f'线性 DQN 骨架训练后: 贪心走 {steps} 步到终点')
print('对照伪代码: 回放/采样/更新/探索 四件套全在')


## 6. 目标网络同步频率扫描（🟡 实验）

### 参数 C 的影响

- C 太小（如 1）：目标更新太频繁 ≈ 用自己当目标 → 不稳定
- C 太大（如 1000）：目标过时 → 学得慢
- 常见：100~1000 步（Atari 常用 1000）

### 实验

悬崖漫步上迷你 DQN 扫 C ∈ {1, 5, 20, 100}，比较最终累计奖励。

---


In [ ]:
# dqn_sync_sweep.py —— 目标网络同步频率 C 扫描(迷你 DQN, 悬崖)
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
NS, NA = ROWS * COLS, 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step_idx(s, a):
    r, c = divmod(s, COLS)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    s2 = nr * COLS + nc
    if (nr, nc) in CLIFF:
        return START[0] * COLS + START[1], -100.0, False
    if (nr, nc) == GOAL:
        return s2, 0.0, True
    return s2, -1.0, False

def onehot(s):
    x = np.zeros(NS); x[s] = 1.0
    return x

class LinQ:
    def __init__(self, seed=0):
        self.W = np.random.RandomState(seed).randn(NS, NA) * 0.05
    def q(self, s):
        return onehot(s) @ self.W
    def copy_from(self, other):
        self.W = other.W.copy()

def run(C, episodes=300, seed=0):
    rng = np.random.RandomState(seed)
    online, target = LinQ(seed), LinQ(seed + 1)
    target.copy_from(online)
    D, CAP, B, alpha, gamma = [], 2000, 32, 0.02, 0.9
    rewards = []
    step_count = 0
    for ep in range(episodes):
        s = START[0] * COLS + START[1]
        eps = max(0.05, 0.3 * (1 - ep / episodes))
        total, steps = 0.0, 0
        while steps < 200:
            a = rng.randint(NA) if rng.random() < eps else int(np.argmax(online.q(s)))
            s2, r, done = step_idx(s, a)
            D.append((s, a, r, s2, done))
            if len(D) > CAP:
                D.pop(0)
            if len(D) >= B:
                idx = rng.randint(0, len(D), B)
                for i in idx:
                    s_i, a_i, r_i, s2_i, done_i = D[i]
                    y = r_i if done_i else r_i + gamma * np.max(target.q(s2_i))
                    pred = online.q(s_i)[a_i]
                    online.W[:, a_i] += alpha * (y - pred) * onehot(s_i)
            s = s2
            total += r
            steps += 1
            step_count += 1
            if step_count % C == 0:
                target.copy_from(online)
            if done:
                break
        rewards.append(total)
    return np.mean(rewards[-100:])

print(f'{"同步频率 C":<12}{"后100回合均值":>14}')
for C in [1, 5, 20, 100]:
    print(f'{C:<12}{run(C):>14.1f}')
print()
print('预期: C 太小时震荡/不稳, C 适中(20~100)最稳; 该趋势随任务放大')


## 7. 回放池容量与 batch 大小扫描（🟡 实验）

### 直觉

- 池太小：样本相关性强（近段轨迹）→ 去相关失效
- 池太大：旧数据占比高 → 学习变慢（但也更稳）
- batch 大：梯度稳但更新慢；batch 小：噪声大

---


In [ ]:
# dqn_cap_batch.py —— 回放池容量与 batch 扫描(悬崖, 线性 DQN 加速)
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
NS, NA = ROWS * COLS, 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step_idx(s, a):
    r, c = divmod(s, COLS)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    s2 = nr * COLS + nc
    if (nr, nc) in CLIFF:
        return START[0] * COLS + START[1], -100.0, False
    if (nr, nc) == GOAL:
        return s2, 0.0, True
    return s2, -1.0, False

def onehot(s):
    x = np.zeros(NS); x[s] = 1.0
    return x

def run(CAP, B, episodes=300, seed=0):
    rng = np.random.RandomState(seed)
    W = rng.randn(NS, NA) * 0.05
    D, alpha, gamma = [], 0.02, 0.9
    rewards = []
    for ep in range(episodes):
        s = START[0] * COLS + START[1]
        eps = max(0.05, 0.3 * (1 - ep / episodes))
        total, steps = 0.0, 0
        while steps < 200:
            a = rng.randint(NA) if rng.random() < eps else int(np.argmax(onehot(s) @ W))
            s2, r, done = step_idx(s, a)
            D.append((s, a, r, s2, done))
            if len(D) > CAP:
                D.pop(0)
            if len(D) >= B:
                idx = rng.randint(0, len(D), B)
                for i in idx:
                    s_i, a_i, r_i, s2_i, done_i = D[i]
                    y = r_i if done_i else r_i + gamma * np.max(onehot(s2_i) @ W)
                    pred = (onehot(s_i) @ W)[a_i]
                    W[:, a_i] += alpha * (y - pred) * onehot(s_i)
            s = s2
            total += r
            steps += 1
            if done:
                break
        rewards.append(total)
    return np.mean(rewards[-100:])

print('回放池容量扫描(batch=32):')
for cap in [100, 500, 2000]:
    print(f'  CAP={cap:<5}: {run(cap, 32):8.1f}')
print('batch 扫描(CAP=2000):')
for b in [8, 32, 128]:
    print(f'  B={b:<4}: {run(2000, b):8.1f}')


## 8. 奖励缩放与奖励剪裁（🟡 训练稳定性）

### 为什么

- 奖励尺度差异大 → 梯度尺度差异大 → 学习率难统一
- Atari 上奖励 clip 到 [-1,1] 是 DQN 标配
- 悬崖漫步的 -100 与 +10 尺度差 10 倍，同样需要小心

### 实验

把悬崖奖励整体缩小（÷10）训练，对比原始尺度的学习曲线。

---


In [ ]:
# reward_scale.py —— 奖励缩放对比(悬崖, 迷你 DQN)
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
NS, NA = ROWS * COLS, 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step_idx(s, a, scale):
    r, c = divmod(s, COLS)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    s2 = nr * COLS + nc
    if (nr, nc) in CLIFF:
        return START[0] * COLS + START[1], -100.0 * scale, False
    if (nr, nc) == GOAL:
        return s2, 0.0, True
    return s2, -1.0 * scale, False

def onehot(s):
    x = np.zeros(NS); x[s] = 1.0
    return x

def run(scale, episodes=300, seed=0):
    rng = np.random.RandomState(seed)
    W = rng.randn(NS, NA) * 0.05
    D, CAP, B, alpha, gamma = [], 2000, 32, 0.02, 0.9
    rewards = []
    for ep in range(episodes):
        s = START[0] * COLS + START[1]
        eps = max(0.05, 0.3 * (1 - ep / episodes))
        total, steps = 0.0, 0
        while steps < 200:
            a = rng.randint(NA) if rng.random() < eps else int(np.argmax(onehot(s) @ W))
            s2, r, done = step_idx(s, a, scale)
            D.append((s, a, r, s2, done))
            if len(D) > CAP:
                D.pop(0)
            if len(D) >= B:
                idx = rng.randint(0, len(D), B)
                for i in idx:
                    s_i, a_i, r_i, s2_i, done_i = D[i]
                    y = r_i if done_i else r_i + gamma * np.max(onehot(s2_i) @ W)
                    pred = (onehot(s_i) @ W)[a_i]
                    W[:, a_i] += alpha * (y - pred) * onehot(s_i)
            s = s2
            total += r
            steps += 1
            if done:
                break
        rewards.append(total)
    return np.mean(rewards[-100:])

for scale in [1.0, 0.1]:
    print(f'奖励尺度 ×{scale}: 后100回合均值 {run(scale):.1f} (按原始奖励计量)')
print()
print('提示: 实际使用中固定尺度 + 适当 α; 比"裸奔大尺度奖励"更稳')


## 9. DDQN 手写实现（🔴 必做实验）

### 关键差异（背）

```text
DQN : y = r + γ·max_a' Q_target(s', a')
DDQN: a* = argmax_a' Q_online(s', a')        ← 用 online 选动作
      y = r + γ·Q_target(s', a*)             ← 用 target 给估值
```

悬崖漫步 + 有噪声奖励时，DDQN 的 Q 估值更贴近真实（虚高更少）。

---


In [ ]:
# ddqn_mini.py —— 迷你 DQN vs DDQN: 悬崖漫步估值偏差对比
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
NS, NA = ROWS * COLS, 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step_idx(s, a, noise=0.0):
    r, c = divmod(s, COLS)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    s2 = nr * COLS + nc
    extra = noise * np.random.randn()
    if (nr, nc) in CLIFF:
        return START[0] * COLS + START[1], -100.0 + extra, False
    if (nr, nc) == GOAL:
        return s2, 0.0, True
    return s2, -1.0 + extra, False

def onehot(s):
    x = np.zeros(NS); x[s] = 1.0
    return x

class LinQ:
    def __init__(self, seed=0):
        self.W = np.random.RandomState(seed).randn(NS, NA) * 0.05
    def q(self, s):
        return onehot(s) @ self.W
    def copy_from(self, other):
        self.W = other.W.copy()

def run(double, episodes=400, seed=0, noise=0.5):
    rng = np.random.RandomState(seed)
    online, target = LinQ(seed), LinQ(seed + 1)
    target.copy_from(online)
    D, CAP, B, alpha, gamma = [], 2000, 32, 0.02, 0.9
    q_est = 0.0
    step_count = 0
    for ep in range(episodes):
        s = START[0] * COLS + START[1]
        eps = max(0.05, 0.3 * (1 - ep / episodes))
        steps = 0
        while steps < 200:
            a = rng.randint(NA) if rng.random() < eps else int(np.argmax(online.q(s)))
            s2, r, done = step_idx(s, a, noise)
            D.append((s, a, r, s2, done))
            if len(D) > CAP:
                D.pop(0)
            if len(D) >= B:
                idx = rng.randint(0, len(D), B)
                for i in idx:
                    s_i, a_i, r_i, s2_i, done_i = D[i]
                    if done_i:
                        y = r_i
                    elif double:
                        a_star = int(np.argmax(online.q(s2_i)))     # online 选动作
                        y = r_i + gamma * target.q(s2_i)[a_star]     # target 估值
                    else:
                        y = r_i + gamma * np.max(target.q(s2_i))
                    pred = online.q(s_i)[a_i]
                    online.W[:, a_i] += alpha * (y - pred) * onehot(s_i)
            s = s2
            steps += 1
            step_count += 1
            if step_count % 20 == 0:
                target.copy_from(online)
            if done:
                break
    # 报告起点 Q 估值(虚高程度)
    return online.q(START[0] * COLS + START[1]).max()

q_dqn = run(double=False)
q_ddqn = run(double=True)
print(f'DQN  起点 Q 估值: {q_dqn:.2f}  (虚高: 与真实最优比较)')
print(f'DDQN 起点 Q 估值: {q_ddqn:.2f}')
print()
print('预期: DDQN 估值更保守(更低) → 说明高估被抑制')


## 10. Dueling 网络手写结构（🟡 实现）

### 结构

```text
输入 x ──> 共享特征层
           ├── V 分支:  → V(s)                     (1 个数)
           └── A 分支:  → A(s,a)                   (|A| 个数)
输出 Q(s,a) = V(s) + A(s,a) - mean_a A(s,a)   (均值版, 可辨识)
```

### 为什么用均值减（背）

> 直接 $V + A$ 不唯一（给 V 加常数、A 减常数不变）；减 $\max$ 或 $\mathrm{mean}$ 让分解唯一。均值版训练更稳（Dueling 论文推荐）。

---


In [ ]:
# dueling_mini.py —— Dueling 网络手写: V + A - mean(A) 结构
import numpy as np

NS, NA = 16, 4
rng = np.random.RandomState(0)

# 共享特征层(线性) + 双分支
W_feat = rng.randn(NS, 8) * 0.1
W_V = rng.randn(8, 1) * 0.1
W_A = rng.randn(8, NA) * 0.1

def dueling_q(s):
    h = np.maximum(0, s @ W_feat)          # 特征层 ReLU
    V = h @ W_V                            # (1,)
    A = h @ W_A                            # (NA,)
    return (V + A - A.mean()).flatten()    # Q = V + A - mean(A)

x = np.zeros(NS); x[0] = 1.0
q = dueling_q(x)
print('Dueling 输出 Q(状态0):', np.round(q, 3))
print('验证: mean(A)=0 约束下分解唯一 → V 与 A 可辨识')
print()
print('直觉: 若某状态"走哪个动作都一样", A 全接近 0, Q≈V —— 学 V 比学 Q 快')


## 11. 优先经验回放 PER（🟡 思路 + 简化实现）

### 核心（背）

按 **TD 误差 $|\delta|$** 给样本加权：

$$P(i) \propto (|\delta_i| + \epsilon)^{\rho}$$

- 误差大 = 预测偏差大 = 更值得学 → 优先采样
- 同时用**重要性权重** $w_i \propto (1/P(i))^{\beta}$ 修正采样偏差（否则分布被扭曲）

### 简化实验

悬崖漫步上用"TD 误差排序采样"替代均匀采样，对比学习速度。

---


In [ ]:
# per_mini.py —— 简化 PER: 按 TD 误差概率采样 vs 均匀采样
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
NS, NA = ROWS * COLS, 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step_idx(s, a):
    r, c = divmod(s, COLS)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    s2 = nr * COLS + nc
    if (nr, nc) in CLIFF:
        return START[0] * COLS + START[1], -100.0, False
    if (nr, nc) == GOAL:
        return s2, 0.0, True
    return s2, -1.0, False

def onehot(s):
    x = np.zeros(NS); x[s] = 1.0
    return x

def run(use_per, episodes=300, seed=0):
    rng = np.random.RandomState(seed)
    W = rng.randn(NS, NA) * 0.05
    D, CAP, B, alpha, gamma = [], 2000, 32, 0.02, 0.9
    pri = []
    rewards = []
    for ep in range(episodes):
        s = START[0] * COLS + START[1]
        eps = max(0.05, 0.3 * (1 - ep / episodes))
        total, steps = 0.0, 0
        while steps < 200:
            a = rng.randint(NA) if rng.random() < eps else int(np.argmax(onehot(s) @ W))
            s2, r, done = step_idx(s, a)
            D.append((s, a, r, s2, done))
            pri.append(abs(r))                    # 初始优先级(简单近似)
            if len(D) > CAP:
                D.pop(0); pri.pop(0)
            if len(D) >= B:
                if use_per:
                    p = np.array(pri[-len(D):])
                    p = p / p.sum()
                    idx = rng.choice(len(D), B, p=p)
                else:
                    idx = rng.randint(0, len(D), B)
                for i in idx:
                    s_i, a_i, r_i, s2_i, done_i = D[i]
                    y = r_i if done_i else r_i + gamma * np.max(onehot(s2_i) @ W)
                    pred = (onehot(s_i) @ W)[a_i]
                    delta = y - pred
                    W[:, a_i] += alpha * delta * onehot(s_i)
                    if use_per:
                        pri[-len(D) + i] = abs(delta) + 0.01   # 更新优先级
            s = s2
            total += r
            steps += 1
            if done:
                break
        rewards.append(total)
    return np.mean(rewards[-100:])

print(f'均匀采样:   后100回合均值 {run(False):.1f}')
print(f'PER(简化):  后100回合均值 {run(True):.1f}')
print()
print('完整 PER 还需重要性权重修正; 此处展示"误差大样本更受重视"的核心思想')


## 12. 训练稳定性检查清单与故障排查（🔴 工程向）

### 如果 DQN 不收敛，按顺序排查（背）

| 症状 | 排查方向 |
|---|---|
| 奖励一直很低 | ε 太大？奖励尺度？网络没学到（lr 小/层浅） |
| 训练震荡发散 | 目标网络同步太频繁？lr 太大？回放池太小？ |
| Q 值爆炸 | 梯度剪裁缺失；奖励未 clip；lr 过大 |
| 学习停滞 | 网络容量不足；特征编码差；回放池过旧 |
| 表现忽好忽坏 | ε 退火过快；seed 方差（多 seed 取中位数报告） |

### 通用口诀（背）

```text
先看奖励尺度 → 再看 lr → 三看回放/目标网络 → 四看网络容量 → 最后多 seed 平均
```

---


## 13. 面试追问速答扩展（🔴）

1. **Q: DQN 为什么不能直接学连续动作？** A: max 需要枚举动作；连续域用 DDPG/TD3/SAC（07 篇）。
2. **Q: 目标网络为什么用"延迟复制"？** A: 让 TD 目标短暂固定，避免移动靶震荡。
3. **Q: 经验回放的缺点？** A: 样本非最新策略分布（off-policy 兼容但分布滞后）；PER 缓解但加复杂度。
4. **Q: Huber 损失 vs MSE？** A: Huber 对离群误差线性惩罚 → 对奖励噪声鲁棒。
5. **Q: DQN 的 Atari 输入怎么编码？** A: 灰度 + 最近 4 帧堆叠 + 归一化到 [0,1]；CNN 提特征。
6. **Q: Double DQN 在哪个步骤消了偏差？** A: 目标计算——selection 与 evaluation 解耦。
7. **Q: 为什么还要用 target 而不是 online？** A: online 自身噪声大；target 更"旧更稳"。
8. **Q: 训练时 reward 为什么 clip？** A: 尺度差异导致梯度主导，clip 到 [-1,1] 后各环境奖励可比。
9. **Q: 如何验证实现正确性？** A: 先在表格小环境验证收敛到 Q*，再放大；对比随机基线。
10. **Q: DQN 在博弈环境的问题？** A: 只学自身策略的 Q，对对手策略变化不敏感 → 需 self-play/NFSP。

---

## 14. 自测清单（04 篇收官）

- [ ] L1：默写 DQN 伪代码与三个稳定性技巧
- [ ] L1：说出 DDQN 与 DQN 目标公式差异
- [ ] L1：解释 Dueling 的 V+A-mean(A) 与可辨识性
- [ ] L2：手写目标网络频率扫描并复现"适中最好"
- [ ] L2：手写简化 PER 并对比均匀采样
- [ ] L3：能把"Q 虚高→DDQN→目标网络→稳定性清单"讲成完整故事


## 15. DQN 家族公式总表（🔴 考前速背）

| 方法 | TD 目标 |
|---|---|
| DQN | $r+\gamma\max_{a'} Q(s',a';\theta^-)$ |
| DDQN | $r+\gamma Q(s',\arg\max_{a'}Q(s',a';\theta);\theta^-)$ |
| Dueling | $Q(s,a)=V(s)+A(s,a)-\mathrm{mean}_{a'}A(s,a')$ |
| N-step | $\sum_{k=0}^{n-1}\gamma^k r_{t+k}+\gamma^n\max Q(s_{t+n})$ |
| PER | $P(i)\propto(|\delta_i|+\epsilon)^{\rho}$，重要性权重修正 |
| 剪裁损失 | Huber($Q_{\theta}(s,a)-y$) |


## 16. 深度 RL 稳定训练检查单（🔴 考前一条条过）

```text
□ 奖励 clip 到 [-1,1]（或归一化）
□ 梯度 clip 全局范数 ≤ 5~10
□ 目标网络同步 C=1000 步（Atari）
□ 回放池 1e5~1e6, batch 32
□ ε 退火 1.0→0.01
□ γ=0.99, Adam lr=1e-4~3e-4
□ 多 seed(≥3) 报告中位数
□ 若发散: 检查 lr→回放→目标网络→网络容量
```


## 17. 面试表达模板：DQN 为什么稳定（🔴 30 秒版）

> "DQN 把 Q-learning 搬上神经网络，但直接上会崩。三个关键设计：
> ① 经验回放——采样去时间相关，训练稳定且数据复用；
> ② 目标网络——TD 目标里的价值是延迟更新的快照，避免追逐移动靶；
> ③ Huber/剪裁——对离群 TD 误差鲁棒。
> 进一步用 DDQN 消选择偏差、Dueling 拆 V 与 A、PER 优先学难的样本，合起来就是 Rainbow。"

---


## 18. 考前 3 分钟速背（🔴 DQN 篇）

```text
三件套: 回放(去相关+复用) / 目标网络(固定监督) / 剪裁(限梯度)
DDQN:  online 选动作, target 估值 → 治高估
Dueling: Q=V+A-mean(A) → 学 V 更快
PER: 按 |δ| 加权采样 + 重要性修正
N-step: 偏差方差旋钮 (Rainbow 组件)
Atari 配方: 4帧堆叠 + CNN + clip[-1,1] + γ0.99 + ε退火
```


## 19. 常见追问组合演练（🔴 面试对练）

- **A**: "目标网络多长时间同步？" → 每 C 步硬拷贝(1e3~1e4) 或软更新 τ=1e-3
- **B**: "回放池多大？" → 1e5~1e6；太旧数据拖慢，太小相关强
- **C**: "为什么 clip 奖励？" → 尺度统一防梯度主导
- **D**: "Dueling 和 DDQN 能叠吗？" → 能（Rainbow 就是这么组合的）


In [ ]:
# comp_check.py —— 诊断实验: 不同组件组合的收益(悬崖, 线性 DQN)
import numpy as np

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
NS, NA = ROWS * COLS, 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step_idx(s, a):
    r, c = divmod(s, COLS)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    s2 = nr * COLS + nc
    if (nr, nc) in CLIFF:
        return START[0] * COLS + START[1], -100.0, False
    if (nr, nc) == GOAL:
        return s2, 0.0, True
    return s2, -1.0, False

def onehot(s):
    x = np.zeros(NS); x[s] = 1.0
    return x

def run(variant, episodes=300, seed=0):
    rng = np.random.RandomState(seed)
    W = rng.randn(NS, NA) * 0.05
    n2 = W.shape[1]
    # Dueling 改造: Q = V + A - mean(A); 用同一矩 W 拆两套权重近似演示
    Wv = rng.randn(NS, 1) * 0.05
    Wa = rng.randn(NS, NA) * 0.05
    D, CAP, B, alpha, gamma = [], 2000, 32, 0.02, 0.9
    rewards = []
    for ep in range(episodes):
        s = START[0] * COLS + START[1]
        eps = max(0.05, 0.3 * (1 - ep / episodes))
        total, steps = 0.0, 0
        while steps < 200:
            if variant == 'dueling':
                h = onehot(s)
                q = (h @ Wv + (h @ Wa) - (h @ Wa).mean()).flatten()
                a = rng.randint(NA) if rng.random() < eps else int(np.argmax(q))
            else:
                a = rng.randint(NA) if rng.random() < eps else int(np.argmax(onehot(s) @ W))
            s2, r, done = step_idx(s, a)
            D.append((s, a, r, s2, done))
            if len(D) > CAP:
                D.pop(0)
            if len(D) >= B:
                idx = rng.randint(0, len(D), B)
                for i in idx:
                    s_i, a_i, r_i, s2_i, done_i = D[i]
                    y = r_i if done_i else r_i + gamma * np.max(onehot(s2_i) @ W)
                    pred = (onehot(s_i) @ W)[a_i]
                    W[:, a_i] += alpha * (y - pred) * onehot(s_i)
                    if variant == 'dueling':
                        # 同步更新 dueling 分量(简化的共享特征近似)
                        Wv += 0.5 * alpha * (y - pred) * onehot(s_i).reshape(-1, 1)
                        Wa[:, a_i] += 0.5 * alpha * (y - pred) * onehot(s_i)
            s = s2
            total += r
            steps += 1
            if done:
                break
        rewards.append(total)
    return np.mean(rewards[-100:])

print(f'{"变体":<14}{"后100回合均值":>14}')
print(f'{"标准DQN":<14}{run("std"):>14.1f}')
print(f'{"Dueling近似":<14}{run("dueling"):>14.1f}')
print()
print('注: 线性近似下差距较小; 真实深度网络+Dueling 优势更明显(学 V 分支加速)')


## 20. 项目脑暴：从零训练一个 Mini-Atari（🟢 实操路线）

```text
环境: 自写 4x4 网格(现有) → 升级像素输入(28x28) 
网络: 2层 CNN + MLP head (numpy 或 torch)
训练: DQN 三件套 + DDQN + Dueling + PER(逐步叠加)
评估: 步数/奖励中位数 + 5 seeds
产出: 学习曲线图 + 组件消融表 → 直接可写进简历项目
```

> 面试官问"做过什么 RL 项目"，上面的执行清单比"我玩过 gym"强 100 倍。


## 21. 深度化：从表格到 CNN 的跳跃（🟡 特征表示）

### 为什么 CNN

Atari 输入是 84×84×4 的帧堆叠（灰度+最近4帧）：空间局部相关 → CNN 提特征远比 MLP 高效。

```text
输入 84×84×4
→ Conv(8×8, stride4, 32) → ReLU
→ Conv(4×4, stride2, 64) → ReLU
→ Conv(3×3, stride1, 64) → ReLU
→ FC(512) → ReLU
→ 输出 |A|=18 个 Q 值
```

> 面试问"DQN 网络结构"——报这个 Atari 经典结构即可（Nature 论文参数）。


In [ ]:
# cnn_arch.py —— DQN CNN 参数量手算(理解"参数量级")
def conv_out(h, k, s, p=0):
    return (h - k + 2 * p) // s + 1

h, w = 84, 84
for name, k, s, ch_out in [('C1', 8, 4, 32), ('C2', 4, 2, 64), ('C3', 3, 1, 64)]:
    h, w = conv_out(h, k, s), conv_out(w, k, s)
    print(f'{name}: 输出 {h}x{w}x{ch_out}')
feat = h * w * 64
fc = 512
print(f'FC 参数量 ≈ {feat // 10000}万 (输入特征 {feat})')
print('→ 全连接层是大头; 这也是为什么要先 CN 降维')


## 22. 样本效率话题（🟡 面试高频延伸）

### 为什么 RL 样本效率低（背三点）

1. **非 i.i.d.**：相邻样本强相关（回放池解决）
2. **延迟奖励**：信用分配难，信号稀疏
3. **探索代价**：试错的样本大多"无效"

### 提升手段（背五点）

1. 经验回放 + PER
2. N-step / GAE（信息利用更充分）
3. Model-based（学模型做想象训练，如 Dreamer）
4. 模仿学习预热（先行为克隆再 RL）
5. 重参数化技巧 / 分布式 RL（Ape-X/IMPALA 吞吐）

> 面试金句："RL 的样本效率落后监督学习 1~2 个数量级，工程上靠 回放+多步+MBRL+并行 四个手段追"。


In [ ]:
# sample_efficiency.py —— 不同方法达到同一性能所需交互次数(示意图)
import numpy as np

# 示例: 一张"学习曲线对比"数值(非真实基准, 教学示意)
fig_data = {
    '随机基线': np.array([60, 59, 61, 60, 60, 60]),
    'Q-learning': np.array([-80, -40, -18, -9, -4, -2]),
    'DQN':        np.array([-60, -20, -8, -3, -1, -1]),
    'DQN+PER':    np.array([-40, -10, -3, -1, -1, -1]),
}
for name, curve in fig_data.items():
    print(f'{name:<12}: ' + ' | '.join(f'{v:+.0f}' for v in curve))
print()
print('→ 越先进的方法, 更快逼近最优奖励(更少样本); 报告时附这条曲线最有说服力')


## 23. 面对"为什么还要传统方法"类问题（🟡 辩证回答）

| 对比 | 表格方法 | 深度方法 |
|---|---|---|
| 可解释 | ✅ 可查表 | 难解释 |
| 连续/大状态 | ❌ | ✅ |
| 收敛保证 | ✅ 理论 | 无 |
| 样本效率(小问题) | ✅ | 高开销 |
| 部署成本 | 轻 | 重(网络) |

> **回答模板**："小规模、可解释、理论安全 → 表格；大规模、连续、泛化 → 深度；两者是连续谱不是二选一"。


## 24. 章末寄语（🟢）

DQN 三件套 + 三个改进（DDQN/Dueling/PER）是"深度价值方法"的全部武器。下一章换赛道：不学 Q 而直接学策略——连续动作的钥匙。


## 25. 深度价值方法的发展脉络（🟢 时间线）

```text
2013  DQN(Atari 超人类)      ← 开山
2015  Nature DQN(目标网络)   ← 定型
2016  DDQN / Dueling / PER   ← 三件改进
2017  Rainbow / N-step       ← 集大成
2018  Distributed(DQN)       ← 分布视角
2020+  Offline RL / 世界模型  ← 新赛道
```

> 面试被问"RL 里程碑"，报 DQN→DDQN→Rainbow 三代就够。


In [ ]:
# dqn_timeline.py —— 里程碑打印 + 一句话记忆
events = [
    ('2013', 'DQN', 'CNN+回放，Atari 部分超人类'),
    ('2015', 'Nature DQN', '加目标网络，稳定训练'),
    ('2016', 'DDQN', '选/估解耦，治高估'),
    ('2016', 'Dueling', '拆 V 与 A，学 V 更快'),
    ('2016', 'PER', '按 TD 误差加权采样'),
    ('2017', 'Rainbow', '六件套合体，Atari SOTA'),
    ('2018', 'Ape-X/IMPALA', '分布式吞吐'),
    ('2020+', 'Offline RL', '只用数据集训练'),
]
for y, n, d in events:
    print(f'{y}  {n:<12} {d}')


## 26. 对比表格：Q-learning→DQN 的逐项迁移（🔴 面试表述）

| 组件 | 表格 Q | DQN |
|---|---|---|
| Q 表示 | 表 | 网络 |
| 更新 | 查表增量 | 梯度下降 |
| 目标 | γmaxQ[表] | γmaxQ(目标网络) |
| 数据 | 在线单步 | 回放池采样 |
| 探索 | ε-greedy | ε-greedy |
| 收敛 | 理论保证 | 工程稳定 |

> 表达：**只换表示与优化器，RL 骨架没变**——这句话展示你抓住了本质。


## 27. 章末寄语（🟢）

DQN 家族到此收官。核心资产：三件套 + 三改进 + 一句"RR is combination of MC+DP+function approximation"。


## 28. 数字敏感度速查（🔴 DQN 篇终版）

| 参数 | 典型值 | 范围 |
|---|---|---|
| γ | 0.99 | 0.9~0.999 |
| ε 初始/最终 | 1.0 / 0.01 | 退火 |
| lr(Adam) | 2.5e-4 | 1e-4~3e-4 |
| 回放池 | 1e6 | 1e5~1e6 |
| batch | 32 | 16~256 |
| 目标同步 C | 1000 | 100~1e4 |
| Huber 阈值 | 1.0 | 0.5~10 |


## 29. 章末终版寄语（🟢）

> 深度价值方法 = 表格 Q-learning 的骨架 + 神经网络的肌肉 + 三件套的铠甲。骨架不变，变化全在表示与稳定化——抓住这句话，DQN 篇就毕业了。


## 30. 模块衔接（🔴）

价值方法（04 篇）的短板是"连续动作要枚举 max"；下一篇换赛道：**直接学策略分布**（PG），连续动作靠采样——05 篇见。


## 31. DQN 家族变种谱系与目标逐步变形推导（🔴 面试手推）

### 谱系: Q-learning → DQN → 三大改进

```text
Q-learning:   y = r + γ·max Q(s',·)                        (表格)
DQN:          y = r + γ·max Q_θ⁻(s',·)                     加网络+目标网络
DDQN:         a* = argmax Q_θ(s',·);  y = r + γ·Q_θ⁻(s',a*)  选/估解耦
Dueling:      Q(s,a) = V(s) + A(s,a) - mean_a A(s,a)        拆结构
PER:          P(i) ∝ (|δ_i| + ε)^ρ                          加权采样
N步DQN:       y = Σ_{k<n}γ^k r + γ^n max Q(s_{t+n},·)        多步目标
```

### 关键推导：为什么 DDQN 的 y 能降高估（背）

单表法 $y = r + \gamma\max_{a'}Q(s',a')$ 里，**选择与估值都用同一张表** → 噪声经 max 放大（03 篇 E[max]≥maxE 推导）。
DDQN 改成"**online 表选择**、**target 表估值**"：两套噪声独立，$\mathbb{E}[Q_{\theta^-}(s',a^*)]$ 不再被选择噪声污染。

> **面试板书写法**：一行写 DQN y，下一行写 DDQN y，用红笔圈出"argmax 的宿主变了"。


## 32. Dueling 分解的可辨识性推导（🔴 面试手推）

### 问题

直接 $Q = V + A$ 不唯一：给 $V$ 加常数 $c$、给 $A$ 减常数 $c$，$Q$ 不变。

### 三步推导（背）

```text
① 设 Q(s,a) = V(s) + A(s,a) 成立
② 任意常数 c: 令 Ṽ = V + c, Ã = A - c → Ṽ + Ã = V + A = Q
   → 分解有无穷多个 (V, A) 组合, 不可辨识
③ 加约束: Σ_a A(s,a) = 0 (或 A_max = 0) 固定冗余自由度
   → 唯一分解: V(s) = max_a Q(s,a) (或均值), A(s,a) = Q - V
   实践中用"减均值": Q = V + A - mean(A) 更稳(梯度路径更短)
```

> **面试表达**："减均值版本等价于'以 A 的均值为 0 作基准'——工程上比减 max 更平滑，Dueling 论文推荐。"


In [ ]:
# fig_target_sync.py —— 过程图①: 目标网络"冻结-同步"机制示意
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

steps = np.arange(0, 100)
online = 2.0 + 1.5 * np.sin(steps / 6) + 0.2 * steps / 100     # 在线网络缓慢爬升
target = np.array([online[min(st // 25 * 25, 99)] for st in steps])  # 每25步硬拷贝

fig, ax = plt.subplots(figsize=(7.5, 4.0), dpi=150)
ax.plot(steps, online, lw=2, color='crimson', label='online 网络(每步更新)')
ax.plot(steps, target, lw=2, ls='--', color='royalblue', label='target 网络(冻结→同步)')
ax.annotate('硬拷贝 θ⁻ ← θ', xy=(25, target[25]), xytext=(30, 2.6),
            arrowprops=dict(arrowstyle='->'), fontsize=9)
ax.set_xlabel('训练步数'); ax.set_ylabel('Q 估计')
ax.set_title('目标网络机制: 中间冻结住监督信号, 每 C 步同步一次')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join('autumn-recruit-algo', '07-强化学习', '教学', 'images')
if os.path.basename(os.getcwd()) == '教学':
    out = os.path.join(os.getcwd(), 'images')
os.makedirs(out, exist_ok=True)
fig.savefig(os.path.join(out, 'dqn_target_sync.png'), dpi=150)
print('已保存 images/dqn_target_sync.png')
plt.close(fig)
plt.show()


In [ ]:
# fig_ddqn_bias.py —— 过程图②: DDQN vs DQN 起点 Q 估值对比(悬崖, 噪声奖励)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

ROWS, COLS = 4, 11
START, GOAL = (3, 0), (3, 10)
CLIFF = [(3, c) for c in range(1, 10)]
NS, NA = ROWS * COLS, 4
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def step_idx(s, a, noise=0.5):
    r, c = divmod(s, COLS)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    nr, nc = max(0, min(ROWS - 1, nr)), max(0, min(COLS - 1, nc))
    s2 = nr * COLS + nc
    extra = noise * np.random.randn()
    if (nr, nc) in CLIFF:
        return START[0] * COLS + START[1], -100.0 + extra, False
    if (nr, nc) == GOAL:
        return s2, 0.0, True
    return s2, -1.0 + extra, False

def onehot(s):
    x = np.zeros(NS); x[s] = 1.0
    return x

class LinQ:
    def __init__(self, seed=0):
        self.W = np.random.RandomState(seed).randn(NS, NA) * 0.05
    def q(self, s):
        return onehot(s) @ self.W
    def copy_from(self, other):
        self.W = other.W.copy()

def run(double, seed=0, episodes=300):
    rng = np.random.RandomState(seed)
    online, target = LinQ(seed), LinQ(seed + 1)
    target.copy_from(online)
    D, CAP, B, alpha, gamma = [], 2000, 32, 0.02, 0.9
    step_count = 0
    for ep in range(episodes):
        s = START[0] * COLS + START[1]
        eps = max(0.05, 0.3 * (1 - ep / episodes))
        steps = 0
        while steps < 200:
            a = rng.randint(NA) if rng.random() < eps else int(np.argmax(online.q(s)))
            s2, r, done = step_idx(s, a)
            D.append((s, a, r, s2, done))
            if len(D) > CAP:
                D.pop(0)
            if len(D) >= B:
                idx = rng.randint(0, len(D), B)
                for i in idx:
                    s_i, a_i, r_i, s2_i, done_i = D[i]
                    if done_i:
                        y = r_i
                    elif double:
                        a_star = int(np.argmax(online.q(s2_i)))
                        y = r_i + gamma * target.q(s2_i)[a_star]
                    else:
                        y = r_i + gamma * np.max(target.q(s2_i))
                    pred = online.q(s_i)[a_i]
                    online.W[:, a_i] += alpha * (y - pred) * onehot(s_i)
            s = s2
            steps += 1
            step_count += 1
            if step_count % 20 == 0:
                target.copy_from(online)
            if done:
                break
    return online.q(START[0] * COLS + START[1]).max()

vals = {'DQN': [run(False, seed=s) for s in range(4)],
        'DDQN': [run(True, seed=s) for s in range(4)]}
means = {k: np.mean(v) for k, v in vals.items()}
stds = {k: np.std(v) for k, v in vals.items()}

fig, ax = plt.subplots(figsize=(6.5, 4.0), dpi=150)
b = ax.bar(list(means.keys()), list(means.values()), yerr=list(stds.values()),
           color=['crimson', 'royalblue'], capsize=6, alpha=0.85)
ax.set_ylabel('起点 Q 估计(越贴近真实越低)')
ax.set_title('DDQN 抑制 Q 值虚高(噪声奖励悬崖环境)')
for rect, v in zip(b, means.values()):
    ax.text(rect.get_x() + rect.get_width()/2, rect.get_height(),
            f'{v:.2f}', ha='center', va='bottom')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
out = os.path.join('autumn-recruit-algo', '07-强化学习', '教学', 'images')
if os.path.basename(os.getcwd()) == '教学':
    out = os.path.join(os.getcwd(), 'images')
os.makedirs(out, exist_ok=True)
fig.savefig(os.path.join(out, 'ddqn_bias.png'), dpi=150)
print('已保存 images/ddqn_bias.png')
plt.close(fig)
plt.show()


## 📸 本章推导配图（教学/images/）

![目标网络机制: 冻结监督信号, 每 C 步硬拷贝同步](images/dqn_target_sync.png)

> 目标网络机制: 冻结监督信号, 每 C 步硬拷贝同步

![DDQN 抑制 Q 值虚高（噪声奖励悬崖环境）](images/ddqn_bias.png)

> DDQN 抑制 Q 值虚高（噪声奖励悬崖环境）
